# Experiment: Case-001 public outreach branch routing gate

Objective: smoke-test that public outreach asks for role, domain, and public source while routing unsafe replies to blocked states.

In [ ]:
from __future__ import annotations

LABEL = "case001_public_outreach_branch_routing_ready_general_only"
BRANCH_STATE = "branch_review_handoff_packet_incomplete"

public_post = " ".join(
    [
        "Nakafa Lab has not found a cure and is not asking for public "
        "treatment advice.",
        "The current need is role-bounded review of a thalassemia "
        "branch-review packet.",
        "Reply with your role, narrow domain, and any public source or registry ID.",
        "Do not post raw records, eligibility claims, referral instructions, "
        "or treatment suggestions.",
    ]
)

required_terms = [
    "role",
    "narrow domain",
    "public source",
    "registry id",
    "not asking for public treatment advice",
    "do not post raw records",
]
missing = [term for term in required_terms if term not in public_post.lower()]
assert not missing, missing

{"label": LABEL, "branch_state": BRANCH_STATE, "missing_terms": missing}

In [ ]:
def route_reply(text: str) -> str:
    """Route one public reply into a safe public next state."""
    lowered = text.lower()
    if "send me records" in lowered or "upload the lab" in lowered:
        return "blocked_raw_record_request"
    if "dose" in lowered or "stop transfusion" in lowered or "eligible" in lowered:
        return "blocked_treatment_advice"
    if "nct" in lowered or "registry" in lowered:
        return "registry_or_pathway_disambiguation_needed"
    if "assay" in lowered or "hbf readout" in lowered:
        return "assay_feasibility_scope_needed"
    if "software" in lowered or "privacy" in lowered:
        return "repo_task_scope_needed"
    if "fund" in lowered or "access map" in lowered:
        return "access_mapping_scope_needed"
    if "tafsir" in lowered or "islamic ethics" in lowered:
        return "ethics_scope_needed"
    if "hematologist" in lowered or "transfusion" in lowered:
        return "role_scope_check_needed"
    return "share_to_qualified_roles"


samples = {
    "hematologist can review transfusion fields": "role_scope_check_needed",
    "NCT06328764 registry context only": "registry_or_pathway_disambiguation_needed",
    "HbF readout assay feasibility": "assay_feasibility_scope_needed",
    "privacy software support": "repo_task_scope_needed",
    "fund access map work": "access_mapping_scope_needed",
    "tafsir boundary review": "ethics_scope_needed",
    "send me records": "blocked_raw_record_request",
    "try this dose": "blocked_treatment_advice",
}
routes = {text: route_reply(text) for text in samples}
assert routes == samples, routes

unsafe_outputs = {
    "diagnosis_output",
    "eligibility_output",
    "treatment_output",
    "referral_output",
    "trial_screening_output",
    "lab_contact_output",
    "sample_routing_output",
}
assert not (set(routes.values()) & unsafe_outputs)
routes

Result: the public call stays role/source/domain based, while raw-record and treatment-advice replies are blocked before follow-up.